# 07 Multimodal Feature Experiments

Tests whether **image–text relationships** add manipulation-detection information: phrase–image consistency, person-name–face consistency and an exploratory expression–sentiment proxy. These features use both modalities even when predicting only image or caption manipulation.

Inputs are matching `02` tables and `04` feature/embedding caches. Images and faces are **not re-encoded**. New text prompts use the encoder corresponding to their cached image/face vectors. Run from `notebooks/` using the project kernel. No `03` outputs are required.

Models fit on `train_sample`; validation is exploratory. Test **features** are exported, but test labels are never evaluated. `05` stays unchanged. The conclusion lists candidates for a later final-model comparison, potentially in `05a`.


## 0. Setup


In [1]:
import os
import sys
import time
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate, best_f1_threshold, combine_fake_score, detection_rate_by_type

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
CACHE_DIR = PROJECT_ROOT / "data" / "interim" / "phrase_clip"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train_sample", "validation", "test"]
RANDOM_STATE = 4012
MAX_PHRASES = 8
CLIP_COLUMNS = ["clip_sim", "clip_sim_txt_norm", "clip_sim_img_norm", "clip_sim_txt_rank"]

sns.set_theme(style="whitegrid")

from itertools import combinations
from datetime import datetime, timezone
from IPython.display import Markdown, display
from src.evaluation import binary_metrics

FACE_CACHE = PROJECT_ROOT / "data/interim/clip"
EXPERIMENT_CACHE = PROJECT_ROOT / "data/interim/multimodal_07"
RESULTS_DIR = PROJECT_ROOT / "results/multimodal_features"
EXPERIMENT_CACHE.mkdir(parents=True, exist_ok=True)
TARGETS = ["image_fake", "text_fake", "is_fake"]
FACE_COLUMNS = ["clip_face_sim_max", "clip_face_sim_min", "clip_face_sim_mean", "clip_face_sim_largest"]
BOOTSTRAPS = 500


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    temporary.replace(path)


timings = {}


## 1. Matching Inputs and Embedding Spaces

Join by unique `row_id`, retain input order, and require identical coverage. Whole-image/phrase comparisons use **ViT-L/14 (768 dimensions)**. Faces/names/expression prompts use **ViT-B/16 (512 dimensions)**; dimensions from different encoders cannot be compared directly.

The baseline for the expanded experiments is eight existing `04` scores: four global image–caption similarities plus four face–caption similarities. This differs from the original phrase experiment's four-global-score baseline; both comparisons are retained and labelled separately.


In [2]:
data = {}
image_vectors = {}
caption_anchors = []

for split in SPLITS:
    rows = pd.read_parquet(PROCESSED_DIR / f"{split}.parquet")
    clip = pd.read_parquet(FEATURES_DIR / f"clip_{split}.parquet")
    emb = pd.read_parquet(FEATURES_DIR / f"clip_emb_{split}.parquet")
    assert rows.row_id.is_unique and clip.row_id.is_unique and emb.row_id.is_unique
    assert set(rows.row_id) == set(clip.row_id) == set(emb.row_id), f"row IDs differ in {split}"
    rows = rows.merge(clip, on="row_id", how="left", validate="one_to_one", sort=False)
    emb = emb.set_index("row_id").loc[rows.row_id]
    img_cols = sorted(c for c in emb if c.startswith("clip_img_"))
    txt_cols = sorted(c for c in emb if c.startswith("clip_txt_"))
    assert len(img_cols) == len(txt_cols) == 768, "Expected ViT-L/14 embeddings from 04"
    image = emb[img_cols].to_numpy(dtype=np.float32, copy=True)
    norms = np.linalg.norm(image, axis=1, keepdims=True)
    assert np.isfinite(image).all() and (norms > 0).all()
    image_vectors[split] = image / norms
    assert np.isfinite(rows[CLIP_COLUMNS].to_numpy()).all()
    for i in range(min(3, len(rows))):
        anchor = emb[txt_cols].iloc[i].to_numpy(dtype=np.float32, copy=True)
        assert np.isfinite(anchor).all() and np.linalg.norm(anchor) > 0
        caption_anchors.append((rows.text.iloc[i], anchor / np.linalg.norm(anchor)))
    data[split] = rows

assert not (set(data["train_sample"].id) & set(data["validation"].id)), "Article overlap"
all_rows = pd.concat(data.values(), ignore_index=True)
captions = pd.Series(all_rows.text.unique(), name="text")
pd.Series({split: len(df) for split, df in data.items()}, name="rows")

for split, rows in data.items():
    face_scores = pd.read_parquet(FEATURES_DIR / f"clip_face_{split}.parquet")
    assert face_scores.row_id.is_unique and set(face_scores.row_id) == set(rows.row_id)
    data[split] = rows.merge(face_scores[["row_id"] + FACE_COLUMNS], on="row_id", validate="one_to_one", sort=False)
    # No-face summaries from 04 can be NaN: the later availability controls distinguish them.
    data[split][FACE_COLUMNS] = data[split][FACE_COLUMNS].fillna(0.)
    assert np.isfinite(data[split][FACE_COLUMNS].to_numpy()).all()

input_files = [PROCESSED_DIR / f"{s}.parquet" for s in SPLITS]
input_files += [FEATURES_DIR / f"{g}_{s}.parquet" for g in ["clip", "clip_emb", "clip_face"] for s in SPLITS]
data_fingerprints = {str(p.relative_to(PROJECT_ROOT)): file_sha256(p) for p in input_files}


## 2. Phrase–Image Consistency

**Hypothesis:** individual activities or settings may disagree with an image despite plausible whole-caption similarity.

Extract up to eight unique noun chunks and eight verb-centred phrases per caption. For each category, save similarity minimum/maximum/mean/standard deviation, whole-caption minus phrase-mean similarity, count and missing indicator: **14 columns**. Missing categories are zero-filled with an indicator. Counts are text-only controls, not evidence of cross-modal agreement.

The following calculation and its caches preserve the completed experiment. spaCy locates phrases without using labels. Parsing mistakes and genuine but invisible context can create low similarities.


In [3]:
def phrases_from_doc(doc):
    nouns = list(dict.fromkeys(chunk.text for chunk in doc.noun_chunks))[:MAX_PHRASES]
    verbs = []
    for token in doc:
        if token.pos_ != "VERB":
            continue
        positions = [token.i]
        for child in token.children:
            if child.dep_ in {"aux", "auxpass", "neg", "prt", "advmod", "obj", "dobj", "attr", "prep"}:
                positions.extend(word.i for word in child.subtree)
        lo, hi = min(positions), max(positions) + 1
        if hi - lo <= 20:
            verbs.append(doc[lo:hi].text)
    return {"noun": nouns, "verb": list(dict.fromkeys(verbs))[:MAX_PHRASES]}


def build_phrase_features(rows, image_vectors, phrase_lookup, phrase_vectors):
    records = []
    for i, row in enumerate(rows.itertuples()):
        record = {"row_id": row.row_id}
        for category in ["noun", "verb"]:
            phrases = phrase_lookup[row.text][category]
            similarities = np.asarray([image_vectors[i] @ phrase_vectors[p] for p in phrases])
            prefix = f"mm_phrase_{category}_"
            record[prefix + "count"] = len(phrases)
            record[prefix + "missing"] = int(not phrases)
            stats = ({"min": similarities.min(), "max": similarities.max(),
                      "mean": similarities.mean(), "std": similarities.std()} if phrases else
                     dict.fromkeys(["min", "max", "mean", "std"], 0.))
            record.update({prefix + name: float(value) for name, value in stats.items()})
            record[prefix + "caption_gap"] = row.clip_sim - stats["mean"] if phrases else 0.
        records.append(record)
    return pd.DataFrame(records)


In [4]:
phrase_stage_start = time.perf_counter()
import spacy

nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "ner"])
phrase_config = {"captions": captions.tolist(), "max_phrases": MAX_PHRASES,
                 "spacy": spacy.__version__, "model": nlp.meta, "method": "noun_chunks_verb_subtrees_v1"}
phrase_key = hashlib.sha256(json.dumps(phrase_config, sort_keys=True).encode()).hexdigest()[:16]
phrase_path = CACHE_DIR / f"caption_phrases_{phrase_key}.json"

if phrase_path.exists():
    phrase_lookup = json.loads(phrase_path.read_text())
    print("Caption phrases loaded from cache")
else:
    start = time.time()
    phrase_lookup = {text: phrases_from_doc(doc)
                     for text, doc in zip(captions, nlp.pipe(captions, batch_size=128))}
    temporary = phrase_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(phrase_lookup))
    temporary.replace(phrase_path)
    print(f"spaCy phrases: {time.time() - start:.0f}s")

assert set(phrase_lookup) == set(captions)
pd.DataFrame([{"text": text, **phrase_lookup[text]} for text in captions.head(5)])


Caption phrases loaded from cache


,text,noun,verb
0,Chanan Reitblat told the court said felt viola...,"[Chanan Reitblat, the court]","[told, said, felt violated, devastated]"
1,David Oyelowo left and Michael Keaton,"[David Oyelowo, Michael Keaton]",[left]
2,No contest LeRoy Neiman presents the boxer Ber...,"[No contest, LeRoy Neiman, the boxer, Bernard ...",[presents the boxer Bernard Hopkins with a por...
3,Rev Jesse Jackson greets attendees before the ...,"[Rev Jesse Jackson greets, the start, the vice...",[]
4,Gingrich talks to Jackson and his father durin...,"[Gingrich talks, Jackson, his father, a break,...",[]


### 2.1 Cached phrase embeddings

Only phrase text is encoded. Completed 1,024-prompt chunks are reused. Re-encoded caption anchors must agree with `04`'s stored caption vectors, preventing a silent encoder-space mismatch.


In [5]:
import torch
import transformers
from src.clip_features import CLIP_NAME, best_device, load_clip, encode_texts

phrases = list(dict.fromkeys(phrase for item in phrase_lookup.values()
                            for category in ["noun", "verb"] for phrase in item[category]))
prompts = list(dict.fromkeys(phrases + [text for text, _ in caption_anchors]))
device = best_device()
model, _, tokenizer = load_clip(device, CLIP_NAME)
encoder_config = {"model": CLIP_NAME, "revision": getattr(model.config, "_commit_hash", None),
                  "torch": str(torch.__version__), "transformers": transformers.__version__,
                  "device": device, "batch_size": 64, "prompts": prompts}
encoder_key = hashlib.sha256(json.dumps(encoder_config, sort_keys=True).encode()).hexdigest()[:16]
embedding_dir = CACHE_DIR / encoder_key
embedding_dir.mkdir(parents=True, exist_ok=True)

parts = []
start = time.time()
for lo in range(0, len(prompts), 1024):
    batch = prompts[lo:lo + 1024]
    path = embedding_dir / f"phrases_{lo:07d}.npy"
    if path.exists():
        part = np.load(path, allow_pickle=False)
    else:
        part = encode_texts(model, tokenizer, batch, device, batch_size=64)
        temporary = path.with_suffix(".tmp")
        with temporary.open("wb") as stream:
            np.save(stream, part, allow_pickle=False)
        temporary.replace(path)
    assert part.shape == (len(batch), 768) and np.isfinite(part).all()
    parts.append(part)
    print(f"{min(lo + 1024, len(prompts)):,}/{len(prompts):,} prompts; {time.time() - start:.0f}s", flush=True)

prompt_emb = np.concatenate(parts)
phrase_vectors = dict(zip(prompts, prompt_emb))
for text, expected in caption_anchors:
    assert np.allclose(phrase_vectors[text], expected, atol=5e-4, rtol=0), "Text encoder differs from 04"
print(f"{len(phrases):,} distinct phrases; {CLIP_NAME} on {device}")
del model


W1118 00:16:51.657000 35911 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

1,024/179,289 prompts; 0s
2,048/179,289 prompts; 0s
3,072/179,289 prompts; 0s
4,096/179,289 prompts; 0s
5,120/179,289 prompts; 0s
6,144/179,289 prompts; 0s
7,168/179,289 prompts; 0s
8,192/179,289 prompts; 0s
9,216/179,289 prompts; 0s
10,240/179,289 prompts; 0s
11,264/179,289 prompts; 0s
12,288/179,289 prompts; 0s
13,312/179,289 prompts; 0s
14,336/179,289 prompts; 0s
15,360/179,289 prompts; 0s
16,384/179,289 prompts; 0s
17,408/179,289 prompts; 0s
18,432/179,289 prompts; 0s
19,456/179,289 prompts; 0s
20,480/179,289 prompts; 0s
21,504/179,289 prompts; 0s
22,528/179,289 prompts; 0s
23,552/179,289 prompts; 0s
24,576/179,289 prompts; 0s
25,600/179,289 prompts; 0s
26,624/179,289 prompts; 0s
27,648/179,289 prompts; 0s
28,672/179,289 prompts; 0s
29,696/179,289 prompts; 0s
30,720/179,289 prompts; 0s
31,744/179,289 prompts; 0s
32,768/179,289 prompts; 0s
33,792/179,289 prompts; 0s
34,816/179,289 prompts; 0s
35,840/179,289 prompts; 0s
36,864/179,289 prompts; 0s
37,888/179,289 prompts; 0s
38,912/179

In [6]:
features = pd.concat([build_phrase_features(rows, image_vectors[split], phrase_lookup, phrase_vectors)
                      for split, rows in data.items()], ignore_index=True)
assert features.row_id.is_unique
assert np.isfinite(features.drop(columns="row_id").to_numpy()).all()
feature_columns = [c for c in features if c != "row_id"]
print(f"{len(feature_columns)} features for {len(features):,} image-caption pairs")
features[feature_columns].describe().round(3)


14 features for 122,774 image-caption pairs


,mm_phrase_noun_count,mm_phrase_noun_missing,mm_phrase_noun_min,mm_phrase_noun_max,mm_phrase_noun_mean,mm_phrase_noun_std,mm_phrase_noun_caption_gap,mm_phrase_verb_count,mm_phrase_verb_missing,mm_phrase_verb_min,mm_phrase_verb_max,mm_phrase_verb_mean,mm_phrase_verb_std,mm_phrase_verb_caption_gap
count,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000
mean,4.732,0.000,0.133,0.230,0.176,0.036,0.079,1.688,0.047,0.170,0.200,0.184,0.014,0.059
std,1.695,0.012,0.034,0.039,0.028,0.016,0.050,0.992,0.212,0.066,0.068,0.064,0.020,0.055
min,0.000,0.000,-0.110,0.000,0.000,0.000,-0.234,0.000,0.000,-0.052,-0.041,-0.041,0.000,-0.210
25%,3.000,0.000,0.114,0.203,0.158,0.025,0.050,1.000,0.000,0.132,0.164,0.153,0.000,0.016
50%,5.000,0.000,0.132,0.231,0.175,0.035,0.082,1.000,0.000,0.162,0.205,0.185,0.000,0.057
75%,6.000,0.000,0.151,0.256,0.194,0.047,0.112,2.000,0.000,0.213,0.246,0.223,0.024,0.097
max,8.000,1.000,0.385,0.422,0.385,0.142,0.269,8.000,1.000,0.433,0.433,0.433,0.149,0.303


In [7]:
def save_features(features, group):
    for split, rows in data.items():
        part = rows[["row_id"]].merge(features, on="row_id", how="left", validate="one_to_one", sort=False)
        assert part.row_id.tolist() == rows.row_id.tolist()
        assert np.isfinite(part.drop(columns="row_id").to_numpy()).all()
        path = FEATURES_DIR / f"{group}_{split}.parquet"
        temporary = path.with_suffix(".tmp.parquet")
        part.to_parquet(temporary, index=False)
        temporary.replace(path)
        print(f"{path.relative_to(PROJECT_ROOT)}  {part.shape}")


save_features(features, "mm_phrase")

phrase_features = features.copy()
phrase_columns = feature_columns.copy()
timings["phrase_stage_seconds_including_cache_reads"] = time.perf_counter() - phrase_stage_start
# Release large temporary arrays before loading the smaller face/text model.
del image_vectors, phrase_vectors, prompt_emb, parts, part, emb, image
import gc
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()


data/features/mm_phrase_train_sample.parquet  (49943, 15)
data/features/mm_phrase_validation.parquet  (22126, 15)
data/features/mm_phrase_test.parquet  (50705, 15)


### 2.2 First recorded validation run

Recorded before this extension, using 49,943 training and 22,126 validation pairs, scaled logistic regression and the **four global CLIP scores**. Values below are rounded notebook observations, not final-test or full-05 results.

| Features | Text-fake AUC | Text-fake F1 | Image-fake AUC | Overall-fake AUC |
|---|---:|---:|---:|---:|
| Global CLIP | 0.820 | 0.622 | 0.546 | 0.667 |
| + Phrase count/availability | 0.820 | 0.626 | 0.547 | 0.667 |
| + Phrase consistency | 0.844 | 0.661 | 0.558 | 0.674 |

**Finding:** phrase similarities added text-manipulation ranking information beyond the structure control. Initial phrase parsing took 97 seconds and the text-encoding loop took 1,126 seconds (about 18.8 minutes); these are recorded component runtimes, not total notebook runtime. However, the overall F1-selected threshold flagged **98.4% of genuine pairs**. Its near-perfect manipulation recall is therefore not useful evidence of practical detection. The new evaluation reports a 1% genuine false-positive operating point as well as F1-selected results.


## 3. Verify Cached Face Alignment

Rebuild `04`'s distinct-image order across `(train_sample, validation, test)`, then its bounding-box order within each image. Legacy `.npy` files have no row keys: matching array lengths alone is insufficient.

Before use, reproduce **all four saved face–caption summaries on every pair** using the cached face/caption vectors. A mismatch stops execution. After verification, store an ordered face-index manifest and file/content fingerprints. Future mismatches are rejected rather than relabelled as trusted caches. Memory mapping and row batches limit memory use; no images are loaded.


In [8]:
def build_face_index(rows):
    images = rows.drop_duplicates("image")[["image", "mtcnn_boxes"]].reset_index(drop=True)
    counts = images.mtcnn_boxes.str.len().to_numpy(dtype=int)
    owner = np.repeat(np.arange(len(images)), counts)
    boxes = np.asarray([b for items in images.mtcnn_boxes for b in items], dtype=float).reshape(-1, 4)
    assert np.isfinite(boxes).all() and ((boxes[:, 2:] - boxes[:, :2]) > 0).all()
    manifest = pd.DataFrame(boxes, columns=["x1", "y1", "x2", "y2"])
    manifest.insert(0, "image", images.image.to_numpy()[owner])
    manifest.insert(0, "face_index", np.arange(len(owner)))
    starts = np.r_[0, np.cumsum(counts)[:-1]]
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return images, counts, starts, areas, manifest


def reproduce_face_scores(rows, face_emb, caption_emb, images, counts, starts, areas, batch_size=2000):
    caption_order = pd.Index(rows.text.unique())
    row_image = pd.Index(images.image).get_indexer(rows.image)
    row_caption = caption_order.get_indexer(rows.text)
    assert (row_image >= 0).all() and (row_caption >= 0).all()
    out = np.full((len(rows), 4), np.nan)
    for lo in range(0, len(rows), batch_size):
        hi = min(lo + batch_size, len(rows))
        per_row = counts[row_image[lo:hi]]
        owners = np.repeat(np.arange(hi - lo), per_row)
        indices = np.concatenate([np.arange(starts[j], starts[j] + counts[j]) for j in row_image[lo:hi]])
        if not len(indices):
            continue
        sims = np.einsum("ij,ij->i", face_emb[indices], caption_emb[row_caption[lo:hi][owners]])
        pairs = pd.DataFrame({"owner": owners, "sim": sims, "area": areas[indices]})
        grouped = pairs.groupby("owner", sort=True)
        aggregates = grouped.sim.agg(["max", "min", "mean"])
        out[lo + aggregates.index.to_numpy(), :3] = aggregates.to_numpy()
        largest = pairs.loc[grouped.area.idxmax()].set_index("owner").sim
        out[lo + largest.index.to_numpy(), 3] = largest.to_numpy()
    return out


start = time.perf_counter()
all_rows = pd.concat(data.values(), ignore_index=True)
images, face_counts, face_starts, face_areas, face_manifest = build_face_index(all_rows)
face_emb = np.load(FACE_CACHE / "faces_vitb16.npy", mmap_mode="r", allow_pickle=False)
face_caption_emb = np.load(FACE_CACHE / "captions_vitb16.npy", mmap_mode="r", allow_pickle=False)
assert face_emb.shape == (int(face_counts.sum()), 512)
assert face_caption_emb.shape == (len(captions), 512)
for array in [face_emb, face_caption_emb]:
    for lo in range(0, len(array), 20000):
        block = array[lo:lo + 20000]
        assert np.isfinite(block).all() and np.allclose(np.linalg.norm(block, axis=1), 1., atol=1e-4)

face_identity = {
    "schema": 1, "model": "openai/clip-vit-base-patch16", "margin": 0.2,
    "face_file": file_sha256(FACE_CACHE / "faces_vitb16.npy"),
    "caption_file": file_sha256(FACE_CACHE / "captions_vitb16.npy"),
    "face_order": hashlib.sha256(pd.util.hash_pandas_object(face_manifest, index=False).values.tobytes()).hexdigest(),
    "caption_order": fingerprint(captions.tolist()),
    "row_order": fingerprint(all_rows.row_id.tolist()),
    "reference_scores": {s: data_fingerprints[f"data/features/clip_face_{s}.parquet"] for s in SPLITS},
}
manifest_path = EXPERIMENT_CACHE / "verified_face_manifest.json"
index_path = EXPERIMENT_CACHE / "verified_face_index.parquet"
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == face_identity, "Face cache identity changed; verify its provenance"
    assert index_path.exists() and pd.read_parquet(index_path).equals(face_manifest), "Face-index manifest differs"

actual = reproduce_face_scores(all_rows, face_emb, face_caption_emb, images, face_counts, face_starts, face_areas)
reference = pd.concat([pd.read_parquet(FEATURES_DIR / f"clip_face_{s}.parquet") for s in SPLITS])
reference = reference.set_index("row_id").loc[all_rows.row_id, FACE_COLUMNS].to_numpy()
assert np.allclose(actual, reference, atol=1e-5, rtol=0, equal_nan=True), "Face/caption cache ordering differs from 04"
if not manifest_path.exists():
    temporary = index_path.with_suffix(".tmp.parquet")
    face_manifest.to_parquet(temporary, index=False)
    temporary.replace(index_path)
    write_json(manifest_path, face_identity)
timings["face_alignment_seconds"] = time.perf_counter() - start
row_image_index = pd.Index(images.image).get_indexer(all_rows.image)
print(f"Verified {len(face_emb):,} faces across {len(all_rows):,} pairs against all saved face summaries")


Verified 294,383 faces across 122,774 pairs against all saved face summaries


## 4. Person-Name–Face Consistency

**Hypothesis:** separating names from the caption may expose identity disagreement obscured by activities or background context.

Extract spaCy `PERSON` entities once per unique caption; collapse whitespace and deduplicate case variants within a caption. Each name becomes `a photo of {name}` and is compared with every face using **ViT-B/16**.

For each name, find its best face match; export the **mean/minimum best match** across names and the **mean best-minus-second-best face gap**. Multiple names may choose the same face; this is not identity verification or one-to-one assignment. An unfamiliar name may have a poor match even in a genuine pair.

Also export name count, face count, availability and gap-availability indicators. Missing names/faces give zero summaries; fewer than two faces makes the gap unavailable. Counts are evaluated separately as controls.


In [9]:
def names_from_doc(doc):
    unique = {}
    for entity in doc.ents:
        if entity.label_ == "PERSON":
            name = " ".join(entity.text.split())
            if name:
                unique.setdefault(name.casefold(), name)
    return list(unique.values())


start = time.perf_counter()
name_nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
name_config = {"captions": captions.tolist(), "spacy": spacy.__version__, "model": name_nlp.meta,
               "method": "person_entities_whitespace_casefold_v1"}
name_key = fingerprint(name_config)
name_dir = EXPERIMENT_CACHE / "names" / name_key[:16]
name_dir.mkdir(parents=True, exist_ok=True)
name_lookup = {}
for lo in range(0, len(captions), 2048):
    batch = captions.iloc[lo:lo + 2048].tolist()
    path = name_dir / f"names_{lo:07d}.json"
    if path.exists():
        chunk = json.loads(path.read_text())
    else:
        chunk = {text: names_from_doc(doc) for text, doc in zip(batch, name_nlp.pipe(batch, batch_size=128))}
        write_json(path, chunk)
    assert list(chunk) == batch
    assert all(isinstance(names, list) and all(isinstance(n, str) and n for n in names)
               and len({n.casefold() for n in names}) == len(names) for names in chunk.values())
    name_lookup.update(chunk)
    print(f"Names: {min(lo + 2048, len(captions)):,}/{len(captions):,} captions", flush=True)
del name_nlp
timings["name_extraction_seconds"] = time.perf_counter() - start
pd.Series([len(name_lookup[text]) for text in captions]).describe().rename("names per unique caption")


Names: 2,048/54,588 captions
Names: 4,096/54,588 captions
Names: 6,144/54,588 captions
Names: 8,192/54,588 captions
Names: 10,240/54,588 captions
Names: 12,288/54,588 captions
Names: 14,336/54,588 captions
Names: 16,384/54,588 captions
Names: 18,432/54,588 captions
Names: 20,480/54,588 captions
Names: 22,528/54,588 captions
Names: 24,576/54,588 captions
Names: 26,624/54,588 captions
Names: 28,672/54,588 captions
Names: 30,720/54,588 captions
Names: 32,768/54,588 captions
Names: 34,816/54,588 captions
Names: 36,864/54,588 captions
Names: 38,912/54,588 captions
Names: 40,960/54,588 captions
Names: 43,008/54,588 captions
Names: 45,056/54,588 captions
Names: 47,104/54,588 captions
Names: 49,152/54,588 captions
Names: 51,200/54,588 captions
Names: 53,248/54,588 captions
Names: 54,588/54,588 captions


count    54588.000000
mean         1.304847
std          0.983421
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          8.000000
Name: names per unique caption, dtype: float64

## 5. Expression–Sentiment Agreement (Exploratory)

**Hypothesis:** some edits create unusual combinations of facial affect and caption sentiment. News about a tragedy can legitimately show a smiling person; disagreement is a possible signal, not proof of fraud.

Encode seven fixed expression prompts (happy, sad, angry, fearful, disgusted, surprised, neutral). Each face's **visual valence** is happy similarity minus mean sad/angry/fearful/disgusted similarity. This is a frozen CLIP proxy, not a trained expression classifier or a probability of emotion.

Summarise valence across faces using mean/minimum/maximum/largest-face values. Combine each with VADER compound caption sentiment using a **signed product** and **opposite-sign interaction**, `max(0, -product)`. Retain visual summaries and sentiment separately so a control can test whether interactions add information beyond these individual signals.


### 5.1 Shared cached text encoding

Names, expression prompts and caption anchors share one ViT-B/16 text-encoding pass, after the large phrase model has been released. Chunks are fingerprinted by model revision, prompt order, library versions and settings. Cached chunks are validated before reuse. Encoding caption anchors must agree with the verified `04` ViT-B/16 caption cache.


In [10]:
from src.clip_features import FACE_CLIP_NAME

EXPRESSIONS = ["happy", "sad", "angry", "fearful", "disgusted", "surprised", "neutral"]
expression_prompts = [f"a photo of a person with a {emotion} facial expression" for emotion in EXPRESSIONS]
name_prompts = list(dict.fromkeys(f"a photo of {name}" for names in name_lookup.values() for name in names))
caption_position = {text: i for i, text in enumerate(captions)}
face_anchors = [(text, face_caption_emb[caption_position[text]]) for text, _ in caption_anchors]
face_prompts = list(dict.fromkeys(name_prompts + expression_prompts + [text for text, _ in face_anchors]))


def cached_text_chunks(model, tokenizer, prompts, device, directory, width=512, chunk_size=1024):
    parts = []
    directory.mkdir(parents=True, exist_ok=True)
    for lo in range(0, len(prompts), chunk_size):
        batch = prompts[lo:lo + chunk_size]
        path = directory / f"text_{lo:07d}.npy"
        if path.exists():
            part = np.load(path, allow_pickle=False)
        else:
            part = encode_texts(model, tokenizer, batch, device, batch_size=64)
            assert part.shape == (len(batch), width) and np.isfinite(part).all()
            temporary = path.with_suffix(".tmp")
            with temporary.open("wb") as stream:
                np.save(stream, part, allow_pickle=False)
            temporary.replace(path)
        assert part.shape == (len(batch), width) and np.isfinite(part).all(), "Invalid cached text chunk"
        assert np.allclose(np.linalg.norm(part, axis=1), 1., atol=1e-4)
        parts.append(part)
        print(f"Face-space prompts: {min(lo + chunk_size, len(prompts)):,}/{len(prompts):,}", flush=True)
    return np.concatenate(parts)


start = time.perf_counter()
face_model, _, face_tokenizer = load_clip(device, FACE_CLIP_NAME)
face_encoder_config = {"model": FACE_CLIP_NAME, "revision": getattr(face_model.config, "_commit_hash", None),
                       "torch": str(torch.__version__), "transformers": transformers.__version__,
                       "device": device, "batch_size": 64, "prompts": face_prompts}
face_encoder_key = fingerprint(face_encoder_config)
prompt_dir = EXPERIMENT_CACHE / "face_prompts" / face_encoder_key[:16]
face_prompt_emb = cached_text_chunks(face_model, face_tokenizer, face_prompts, device, prompt_dir)
face_prompt_vectors = dict(zip(face_prompts, face_prompt_emb))
for text, expected in face_anchors:
    assert np.allclose(face_prompt_vectors[text], expected, atol=5e-4, rtol=0), "ViT-B/16 encoder differs from 04"
del face_model
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
timings["shared_name_expression_encoding_seconds"] = time.perf_counter() - start
print(f"{len(name_prompts):,} unique name prompts, seven expression prompts; face-space anchors agree")


config.json:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  599MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

... (3 outputs left out)

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Face-space prompts: 1,024/28,020
Face-space prompts: 2,048/28,020
Face-space prompts: 3,072/28,020
Face-space prompts: 4,096/28,020
Face-space prompts: 5,120/28,020
Face-space prompts: 6,144/28,020
Face-space prompts: 7,168/28,020
Face-space prompts: 8,192/28,020
Face-space prompts: 9,216/28,020
Face-space prompts: 10,240/28,020
Face-space prompts: 11,264/28,020
Face-space prompts: 12,288/28,020
Face-space prompts: 13,312/28,020
Face-space prompts: 14,336/28,020
Face-space prompts: 15,360/28,020
Face-space prompts: 16,384/28,020
Face-space prompts: 17,408/28,020
Face-space prompts: 18,432/28,020
Face-space prompts: 19,456/28,020
Face-space prompts: 20,480/28,020
Face-space prompts: 21,504/28,020
Face-space prompts: 22,528/28,020
Face-space prompts: 23,552/28,020
Face-space prompts: 24,576/28,020
Face-space prompts: 25,600/28,020
Face-space prompts: 26,624/28,020
Face-space prompts: 27,648/28,020
Face-space prompts: 28,020/28,020
28,004 unique name prompts, seven expression prompts; fac

### 5.2 Build and save the two new groups

Feature calculations never access manipulation labels. The same frozen rules apply to every split. Checkpoints include row order, verified face-cache identity, prompt fingerprint and feature-method version; a completed row chunk is reused only under the same identity.


In [11]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


def name_face_record(names, faces, vectors):
    count, n_faces = len(names), len(faces)
    available = bool(count and n_faces)
    gap_available = available and n_faces >= 2
    out = {"name_count": count, "face_count": n_faces, "available": int(available),
           "gap_available": int(gap_available), "best_mean": 0., "best_min": 0., "face_gap_mean": 0.}
    if available:
        similarities = np.stack([vectors[f"a photo of {name}"] for name in names]) @ faces.T
        best = similarities.max(axis=1)
        out.update(best_mean=float(best.mean()), best_min=float(best.min()))
        if gap_available:
            top_two = np.partition(similarities, -2, axis=1)[:, -2:]
            out["face_gap_mean"] = float((top_two.max(axis=1) - top_two.min(axis=1)).mean())
    return {"mm_name_face_" + name: value for name, value in out.items()}


def expression_record(valences, areas, sentiment):
    available = len(valences) > 0
    summaries = ({"mean": float(np.mean(valences)), "min": float(np.min(valences)),
                  "max": float(np.max(valences)), "largest": float(valences[np.argmax(areas)])}
                 if available else dict.fromkeys(["mean", "min", "max", "largest"], 0.))
    out = {"face_count": len(valences), "available": int(available), "sentiment": float(sentiment)}
    for name, value in summaries.items():
        product = value * sentiment
        out.update({"valence_" + name: value, "product_" + name: float(product),
                    "opposite_" + name: float(max(0., -product))})
    return {"mm_expression_" + name: value for name, value in out.items()}


vader = SentimentIntensityAnalyzer()
sentiment_lookup = {text: vader.polarity_scores(text)["compound"] for text in captions}
expression_vectors = np.stack([face_prompt_vectors[p] for p in expression_prompts])
visual_valence = np.empty(len(face_emb), dtype=np.float32)
for lo in range(0, len(face_emb), 10000):
    scores = face_emb[lo:lo + 10000] @ expression_vectors.T
    visual_valence[lo:lo + len(scores)] = scores[:, 0] - scores[:, 1:5].mean(axis=1)
assert np.isfinite(visual_valence).all()

new_feature_config = {"method": "name_best_face_and_clip_valence_v1", "faces": face_identity,
                      "name_key": name_key, "encoder_key": face_encoder_key,
                      "vader_lexicon": fingerprint(vader.lexicon),
                      "rows": fingerprint(all_rows[["row_id", "text", "image"]].values.tolist())}
new_feature_key = fingerprint(new_feature_config)
new_feature_dir = EXPERIMENT_CACHE / "row_features" / new_feature_key[:16]
new_feature_dir.mkdir(parents=True, exist_ok=True)
name_parts, expression_parts = [], []
start = time.perf_counter()
for lo in range(0, len(all_rows), 5000):
    batch = all_rows.iloc[lo:lo + 5000]
    name_path = new_feature_dir / f"names_{lo:07d}.parquet"
    expr_path = new_feature_dir / f"expression_{lo:07d}.parquet"
    if name_path.exists() and expr_path.exists():
        names_part, expr_part = pd.read_parquet(name_path), pd.read_parquet(expr_path)
    else:
        name_records, expr_records = [], []
        for offset, row in enumerate(batch.itertuples()):
            image_index = row_image_index[lo + offset]
            first, count = face_starts[image_index], face_counts[image_index]
            stop = first + count
            name_records.append({"row_id": row.row_id, **name_face_record(
                name_lookup[row.text], face_emb[first:stop], face_prompt_vectors)})
            expr_records.append({"row_id": row.row_id, **expression_record(
                visual_valence[first:stop], face_areas[first:stop], sentiment_lookup[row.text])})
        names_part, expr_part = pd.DataFrame(name_records), pd.DataFrame(expr_records)
        for frame, path in [(names_part, name_path), (expr_part, expr_path)]:
            temporary = path.with_suffix(".tmp.parquet")
            frame.to_parquet(temporary, index=False)
            temporary.replace(path)
    for part in [names_part, expr_part]:
        assert part.row_id.is_unique and part.row_id.tolist() == batch.row_id.tolist()
        assert np.isfinite(part.drop(columns="row_id").to_numpy()).all()
    name_parts.append(names_part)
    expression_parts.append(expr_part)
    print(f"New feature rows: {min(lo + 5000, len(all_rows)):,}/{len(all_rows):,}", flush=True)

name_features = pd.concat(name_parts, ignore_index=True)
expression_features = pd.concat(expression_parts, ignore_index=True)
timings["new_feature_summaries_seconds"] = time.perf_counter() - start
save_features(name_features, "mm_name_face")
save_features(expression_features, "mm_expression")
feature_groups = {"phrases": phrase_features, "names": name_features, "expression": expression_features}
coverage = []
for split, rows in data.items():
    for group, frame in feature_groups.items():
        part = frame.set_index("row_id").loc[rows.row_id]
        available = ((part.mm_phrase_noun_missing == 0) | (part.mm_phrase_verb_missing == 0)
                     if group == "phrases" else part[f"mm_{'name_face' if group == 'names' else 'expression'}_available"] == 1)
        coverage.append({"split": split, "group": group, "rows": len(part), "features": len(part.columns),
                         "available_fraction": float(available.mean()), "finite": bool(np.isfinite(part.to_numpy()).all())})
coverage = pd.DataFrame(coverage)
display(coverage)


New feature rows: 5,000/122,774
New feature rows: 10,000/122,774
New feature rows: 15,000/122,774
New feature rows: 20,000/122,774
New feature rows: 25,000/122,774
New feature rows: 30,000/122,774
New feature rows: 35,000/122,774
New feature rows: 40,000/122,774
New feature rows: 45,000/122,774
New feature rows: 50,000/122,774
New feature rows: 55,000/122,774
New feature rows: 60,000/122,774
New feature rows: 65,000/122,774
New feature rows: 70,000/122,774
New feature rows: 75,000/122,774
New feature rows: 80,000/122,774
New feature rows: 85,000/122,774
New feature rows: 90,000/122,774
New feature rows: 95,000/122,774
New feature rows: 100,000/122,774
New feature rows: 105,000/122,774
New feature rows: 110,000/122,774
New feature rows: 115,000/122,774
New feature rows: 120,000/122,774
New feature rows: 122,774/122,774
data/features/mm_name_face_train_sample.parquet  (49943, 8)
data/features/mm_name_face_validation.parquet  (22126, 8)
data/features/mm_name_face_test.parquet  (50705, 8)


,split,group,rows,features,available_fraction,finite
0,train_sample,phrases,49943,14,1.000000,True
1,train_sample,names,49943,7,0.797029,True
2,train_sample,expression,49943,15,1.000000,True
3,validation,phrases,22126,14,1.000000,True
4,validation,names,22126,7,0.785501,True
5,validation,expression,22126,15,1.000000,True
6,test,phrases,50705,14,1.000000,True
7,test,names,50705,7,0.816310,True
8,test,expression,50705,15,1.000000,True


## 6. What the Models Predict

Each experiment fits **two separate StandardScaler + logistic regression pipelines** (`C=1`, `max_iter=2000`, seed `4012`, no class reweighting), using the same selected features and training rows:

| Output | Target |
|---|---|
| `p_image_fake` | Image edited (`image_fake`) |
| `p_text_fake` | Caption manipulated or swapped (`text_fake`) |
| `p_is_fake` | Derived overall fake-pair score (`is_fake`) |

Both modality labels can be 1. They identify **which modality**, not the editing technique. A model is multimodal because its inputs describe image–text relationships, not because it has a particular number of outputs. No standalone text/image model predictions are required by these experiments.

The existing combination is `p_is_fake = 1 - (1 - p_image_fake) * (1 - p_text_fake)`. This **noisy-OR** assumes independence; correlated outputs mean it need not be a calibrated joint probability. Changing the combination rule is a separate future experiment.

**AUC, PR-AUC and EER use continuous scores and actual labels.** Accuracy, precision, recall and F1 use decisions made by thresholding those scores. AUC assesses ranking; it is not calculated by the noisy-OR formula.


## 7. Controlled Validation Experiments

The **original family** reproduces the four-global-score phrase comparison. The **expanded family** starts from all eight existing global/face similarities, then adds each group, every pair and all three. Controls separate cross-modal similarity from counts, and expression interactions from visual valence/caption sentiment individually.

Every fit uses identical `train_sample` rows and fixed learner settings. No article ID, path, manipulation annotation or genuine-reference caption is a predictor. Counts are explicitly labelled controls. Thresholds are selected and reported on validation, so these operating-point estimates are exploratory, not held-out final-test estimates.

For each output, also choose a threshold using **only genuine validation pairs** with at most 1% flagged; ties are excluded conservatively. Manipulation-type recall is measured using the overall score at that threshold. For `orig`, flagged rate means false-positive rate, not recall.


In [12]:
joined = {}
for split, rows in data.items():
    frame = rows.copy()
    for group in feature_groups.values():
        frame = frame.merge(group, on="row_id", how="left", validate="one_to_one", sort=False)
    assert frame.row_id.tolist() == rows.row_id.tolist()
    joined[split] = frame
train, val = joined["train_sample"], joined["validation"]
group_columns = {name: [c for c in frame if c != "row_id"] for name, frame in feature_groups.items()}
controls = {
    "phrases": [c for c in group_columns["phrases"] if c.endswith(("_count", "_missing"))],
    "names": [c for c in group_columns["names"] if c.endswith(("_count", "_available"))],
    "expression": [c for c in group_columns["expression"] if "_product_" not in c and "_opposite_" not in c],
}
BASE = CLIP_COLUMNS + FACE_COLUMNS
experiments = {
    "Original: global CLIP": CLIP_COLUMNS,
    "Original: global + phrase control": CLIP_COLUMNS + controls["phrases"],
    "Original: global + phrases": CLIP_COLUMNS + group_columns["phrases"],
    "Expanded: baseline": BASE,
}
for size in [1, 2, 3]:
    for groups in combinations(group_columns, size):
        experiments["Expanded: " + " + ".join(groups)] = BASE + [c for g in groups for c in group_columns[g]]
for group in group_columns:
    experiments[f"Control: {group}"] = BASE + controls[group]


def genuine_fpr_threshold(score, genuine, rate=0.01):
    values = np.asarray(score)[np.asarray(genuine, dtype=bool)]
    assert len(values) and np.isfinite(values).all() and 0 <= rate < 1
    allowed = int(np.floor(rate * len(values)))
    # >= threshold excludes all ties at the cutoff, so the allowed count is never exceeded.
    cutoff = np.sort(values)[len(values) - allowed - 1]
    return float(np.nextafter(cutoff, np.inf))


def fit_experiment(columns):
    assert len(columns) == len(set(columns))
    assert np.isfinite(train[columns].to_numpy()).all() and np.isfinite(val[columns].to_numpy()).all()
    predictions = {}
    for target in ["image_fake", "text_fake"]:
        assert train[target].nunique() == val[target].nunique() == 2
        classifier = make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=2000, random_state=RANDOM_STATE))
        classifier.fit(train[columns], train[target])
        predictions[target] = classifier.predict_proba(val[columns])[:, 1]
    predictions["is_fake"] = combine_fake_score(predictions["image_fake"], predictions["text_fake"])
    return predictions


evaluation_start = time.perf_counter()
validation_predictions, metric_rows, low_fpr_rows, type_rows, runtime_rows = {}, [], [], [], []
genuine = val.is_fake.to_numpy() == 0
assert genuine.any()
for experiment, columns in experiments.items():
    start = time.perf_counter()
    scores = fit_experiment(columns)
    validation_predictions[experiment] = scores
    for target, score in scores.items():
        threshold = best_f1_threshold(val[target], score)
        strict_threshold = genuine_fpr_threshold(score, genuine)
        metric_rows.append({"experiment": experiment, "target": target, "n_features": len(columns),
                            **binary_metrics(val[target], score, threshold),
                            "genuine_FPR": float((score[genuine] >= threshold).mean())})
        low_fpr_rows.append({"experiment": experiment, "target": target,
                             **binary_metrics(val[target], score, strict_threshold),
                             "genuine_FPR": float((score[genuine] >= strict_threshold).mean())})
        assert low_fpr_rows[-1]["genuine_FPR"] <= 0.01 + 1e-12
        if target == "is_fake":
            for setting, value in [("validation_F1", threshold), ("validation_genuine_FPR_1pct", strict_threshold)]:
                table = detection_rate_by_type(val, score, value).reset_index()
                table["experiment"], table["operating_point"] = experiment, setting
                type_rows.append(table)
    elapsed = time.perf_counter() - start
    runtime_rows.append({"experiment": experiment, "seconds": elapsed})
    print(f"{experiment}: {len(columns)} features, {elapsed:.1f}s", flush=True)

results = pd.DataFrame(metric_rows)
low_fpr_results = pd.DataFrame(low_fpr_rows)
by_type = pd.concat(type_rows, ignore_index=True)
runtimes = pd.DataFrame(runtime_rows)
timings["validation_fit_and_metrics_seconds"] = time.perf_counter() - evaluation_start
display(results.round(4))
display(low_fpr_results[["experiment", "target", "genuine_FPR", "Recall", "Precision", "F1"]].round(4))
display(by_type[by_type.operating_point == "validation_genuine_FPR_1pct"].round(4))


Original: global CLIP: 4 features, 2.3s
Original: global + phrase control: 8 features, 2.3s
Original: global + phrases: 18 features, 2.4s
Expanded: baseline: 8 features, 2.3s
Expanded: phrases: 22 features, 2.5s
Expanded: names: 15 features, 2.6s
Expanded: expression: 23 features, 2.4s
Expanded: phrases + names: 29 features, 2.7s
Expanded: phrases + expression: 37 features, 2.5s
Expanded: names + expression: 30 features, 2.6s
Expanded: phrases + names + expression: 44 features, 2.8s
Control: phrases: 12 features, 2.4s
Control: names: 12 features, 2.4s
Control: expression: 15 features, 2.4s


,experiment,target,n_features,AUC,PR-AUC,EER,ACC,Precision,Recall,F1,threshold,genuine_FPR
0,Original: global CLIP,image_fake,4,0.5458,0.5711,0.4658,0.5415,0.5409,0.9927,0.7002,0.4474,0.9864
1,Original: global CLIP,text_fake,4,0.8199,0.7075,0.2581,0.7902,0.5954,0.6514,0.6222,0.3252,0.1339
2,Original: global CLIP,is_fake,4,0.6667,0.8142,0.3887,0.6631,0.6643,0.9930,0.7961,0.4632,0.9839
3,Original: global + phrase control,image_fake,8,0.5469,0.5727,0.4658,0.5413,0.5407,0.9925,0.7000,0.4450,0.9868
4,Original: global + phrase control,text_fake,8,0.8205,0.7115,0.2604,0.7959,0.6089,0.6432,0.6256,0.3336,0.1235
5,Original: global + phrase control,is_fake,8,0.6668,0.8147,0.3889,0.6628,0.6642,0.9928,0.7959,0.4618,0.9843
6,Original: global + phrases,image_fake,18,0.5584,0.5827,0.4581,0.5439,0.5423,0.9903,0.7008,0.4274,0.9756
7,Original: global + phrases,text_fake,18,0.8442,0.7491,0.2359,0.8222,0.6678,0.6550,0.6613,0.3643,0.0996
8,Original: global + phrases,is_fake,18,0.6736,0.8216,0.3822,0.6633,0.6644,0.9932,0.7962,0.4509,0.9835
9,Expanded: baseline,image_fake,8,0.5822,0.5946,0.4390,0.5573,0.5509,0.9703,0.7028,0.3979,0.9157


,experiment,target,genuine_FPR,Recall,Precision,F1
0,Original: global CLIP,image_fake,0.0099,0.0424,0.5856,0.0791
1,Original: global CLIP,text_fake,0.0099,0.3292,0.8866,0.4801
2,Original: global CLIP,is_fake,0.0099,0.1480,0.9670,0.2566
3,Original: global + phrase control,image_fake,0.0099,0.0384,0.5902,0.0721
4,Original: global + phrase control,text_fake,0.0099,0.3421,0.8900,0.4943
5,Original: global + phrase control,is_fake,0.0099,0.1507,0.9676,0.2608
6,Original: global + phrases,image_fake,0.0099,0.0235,0.5945,0.0451
7,Original: global + phrases,text_fake,0.0099,0.3965,0.8974,0.5500
8,Original: global + phrases,is_fake,0.0099,0.1718,0.9715,0.2920
9,Expanded: baseline,image_fake,0.0099,0.0230,0.6307,0.0445


,fake_cls,flagged_rate,n,experiment,operating_point
9,orig,0.0099,7473,Original: global CLIP,validation_genuine_FPR_1pct
10,face_attribute,0.0160,3999,Original: global CLIP,validation_genuine_FPR_1pct
11,face_swap,0.0257,4788,Original: global CLIP,validation_genuine_FPR_1pct
12,text_attribute,0.0315,825,Original: global CLIP,validation_genuine_FPR_1pct
13,face_attribute&text_attribute,0.0801,487,Original: global CLIP,validation_genuine_FPR_1pct
...,...,...,...,...,...
247,face_swap&text_attribute,0.0807,533,Control: expression,validation_genuine_FPR_1pct
248,face_attribute&text_attribute,0.0986,487,Control: expression,validation_genuine_FPR_1pct
249,text_swap,0.4424,1894,Control: expression,validation_genuine_FPR_1pct
250,face_swap&text_swap,0.4626,1163,Control: expression,validation_genuine_FPR_1pct


## 8. Paired Article-Level Uncertainty

Use 500 paired bootstrap resamples of **validation articles**, retaining all rows within each sampled article. Identical resamples are used for every experiment. Report percentile 95% intervals for AUC differences against the matching family baseline and relevant controls.

These intervals describe validation-sample uncertainty conditional on the fitted models; models are not refitted inside the bootstrap. Repeated exploratory comparisons are not multiplicity-adjusted. A small positive point estimate alone does not establish a reliable improvement.


In [13]:
def prepare_weighted_auc(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores)
    order = np.argsort(scores, kind="stable")
    starts = np.r_[0, np.flatnonzero(np.diff(scores[order])) + 1]
    return order, starts, labels[order].astype(float)


def weighted_auc(prepared, weights):
    order, starts, labels = prepared
    ordered = np.asarray(weights)[order]
    positive = np.add.reduceat(ordered * labels, starts)
    negative = np.add.reduceat(ordered * (1 - labels), starts)
    denominator = positive.sum() * negative.sum()
    if denominator == 0:
        return np.nan
    return float(np.sum(positive * (np.cumsum(negative) - 0.5 * negative)) / denominator)


comparison_pairs = [("Original: global + phrases", "Original: global CLIP"),
                    ("Original: global + phrases", "Original: global + phrase control")]
comparison_pairs += [(name, "Expanded: baseline") for name in experiments
                     if name.startswith("Expanded:") and name != "Expanded: baseline"]
comparison_pairs += [(f"Expanded: {group}", f"Control: {group}") for group in group_columns]
prepared = {(name, target): prepare_weighted_auc(val[target], score)
            for name, scores in validation_predictions.items() for target, score in scores.items()}
article_codes, articles = pd.factorize(val.id, sort=False)
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_auc = {key: np.empty(BOOTSTRAPS) for key in prepared}
start = time.perf_counter()
for b in range(BOOTSTRAPS):
    multiplicity = np.bincount(rng.integers(0, len(articles), size=len(articles)), minlength=len(articles))
    weights = multiplicity[article_codes]
    for key, prepped in prepared.items():
        bootstrap_auc[key][b] = weighted_auc(prepped, weights)
    if (b + 1) % 100 == 0:
        print(f"Article bootstrap: {b + 1}/{BOOTSTRAPS}", flush=True)

auc_lookup = results.set_index(["experiment", "target"]).AUC
difference_rows = []
for experiment, reference in comparison_pairs:
    for target in TARGETS:
        delta = bootstrap_auc[(experiment, target)] - bootstrap_auc[(reference, target)]
        valid = delta[np.isfinite(delta)]
        assert len(valid) >= 0.95 * BOOTSTRAPS, "Too few bootstrap samples contain both classes"
        lower, upper = np.quantile(valid, [0.025, 0.975])
        difference_rows.append({"experiment": experiment, "reference": reference, "target": target,
                                "AUC_difference": float(auc_lookup.loc[(experiment, target)] - auc_lookup.loc[(reference, target)]),
                                "CI_low": float(lower), "CI_high": float(upper), "valid_resamples": len(valid)})
auc_differences = pd.DataFrame(difference_rows)
timings["article_bootstrap_seconds"] = time.perf_counter() - start
display(auc_differences.round(4))


Article bootstrap: 100/500
Article bootstrap: 200/500
Article bootstrap: 300/500
Article bootstrap: 400/500
Article bootstrap: 500/500


,experiment,reference,target,AUC_difference,CI_low,CI_high,valid_resamples
0,Original: global + phrases,Original: global CLIP,image_fake,0.0125,0.0089,0.0160,500
1,Original: global + phrases,Original: global CLIP,text_fake,0.0243,0.0216,0.0273,500
2,Original: global + phrases,Original: global CLIP,is_fake,0.0069,0.0054,0.0082,500
3,Original: global + phrases,Original: global + phrase control,image_fake,0.0115,0.0079,0.0147,500
4,Original: global + phrases,Original: global + phrase control,text_fake,0.0237,0.0211,0.0268,500
5,Original: global + phrases,Original: global + phrase control,is_fake,0.0068,0.0054,0.0081,500
6,Expanded: phrases,Expanded: baseline,image_fake,0.0051,0.0036,0.0068,500
7,Expanded: phrases,Expanded: baseline,text_fake,0.0201,0.0175,0.0228,500
8,Expanded: phrases,Expanded: baseline,is_fake,0.0084,0.0068,0.0100,500
9,Expanded: names,Expanded: baseline,image_fake,0.0102,0.0072,0.0131,500


## 9. Findings and Reproducible Results

The tables below are populated from the current real-data run. For every group, compare with its matching control and show both ranking and low-false-positive behaviour. Interactions are retained as **candidates**, not automatically selected for the final model.

Candidate labels are an exploratory screen: an AUC interval wholly above zero against the matching control supports further testing; intervals spanning zero are inconclusive, and intervals wholly at/below zero do not support that target. Low-FPR recall changes remain visible even when AUC improves. No claim is made about improvement over the full stacking model.

Compact metrics/configurations are saved under `results/multimodal_features/<run fingerprint>/`. They include code/data/model fingerprints, seed, feature lists, thresholds, runtime, coverage and execution status. Large embeddings/checkpoints remain in ignored `data/` directories. A new code/data configuration gets a different results directory.


In [14]:
hypotheses = {
    "phrases": "Caption details disagree with the image",
    "names": "Named people disagree with visible faces",
    "expression": "Facial valence and caption sentiment interact",
}
strict_lookup = low_fpr_results.set_index(["experiment", "target"])
findings_rows = []
for group in group_columns:
    experiment, reference = f"Expanded: {group}", f"Control: {group}"
    for target in TARGETS:
        comparison = auc_differences.query("experiment == @experiment and reference == @reference and target == @target").iloc[0]
        status = ("candidate" if comparison.CI_low > 0 else
                  "not supported in this experiment" if comparison.CI_high <= 0 else "inconclusive")
        findings_rows.append({"group": group, "hypothesis": hypotheses[group], "target": target,
                              "AUC": float(auc_lookup.loc[(experiment, target)]),
                              "AUC_change_vs_control": comparison.AUC_difference,
                              "CI_low": comparison.CI_low, "CI_high": comparison.CI_high,
                              "recall_at_1pct_genuine_FPR": float(strict_lookup.loc[(experiment, target), "Recall"]),
                              "recall_change_vs_control": float(strict_lookup.loc[(experiment, target), "Recall"] - strict_lookup.loc[(reference, target), "Recall"]),
                              "status": status})
findings = pd.DataFrame(findings_rows)
display(findings.round(4))
display(coverage)
display(runtimes.round(2))

notebook_file = PROJECT_ROOT / "notebooks/07_multimodal_analysis.ipynb"
notebook_sources = [(c["cell_type"], "".join(c["source"])) for c in json.loads(notebook_file.read_text())["cells"]]
import sklearn
from importlib.metadata import version
configuration = {
    "seed": RANDOM_STATE, "classifier": {"name": "StandardScaler + LogisticRegression", "C": 1., "max_iter": 2000, "class_weight": None},
    "fit_split": "train_sample", "evaluation_split": "validation", "test_evaluated": False,
    "bootstrap": {"resamples": BOOTSTRAPS, "unit": "article id", "paired": True, "refit": False},
    "overall_score": "1 - (1 - p_image_fake) * (1 - p_text_fake)",
    "threshold_protocol": "validation F1 and <=1% genuine-validation FPR; >= comparison; ties conservative",
    "code_sha256": fingerprint(notebook_sources),
    "evaluation_helper_sha256": file_sha256(PROJECT_ROOT / "src/evaluation.py"),
    "clip_helper_sha256": file_sha256(PROJECT_ROOT / "src/clip_features.py"),
    "data_sha256": data_fingerprints, "face_identity": face_identity,
    "feature_fingerprints": {"phrases": phrase_key, "names_expression": new_feature_key},
    "phrase_encoder": encoder_config, "face_encoder": face_encoder_config,
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
                 "spacy": spacy.__version__, "vaderSentiment": version("vaderSentiment")},
    "experiments": experiments,
}
# Full ordered prompts are represented by hashes/counts to keep committed configuration compact.
for key in ["phrase_encoder", "face_encoder"]:
    configuration[key] = {**configuration[key], "prompt_sha256": fingerprint(configuration[key]["prompts"]),
                          "prompt_count": len(configuration[key]["prompts"])}
    configuration[key].pop("prompts")
run_key = fingerprint(configuration)[:16]
run_dir = RESULTS_DIR / run_key
run_dir.mkdir(parents=True, exist_ok=True)
write_json(run_dir / "configuration.json", {**configuration, "execution_status": "incomplete_until_metrics_saved"})
tables = {"validation_metrics": results, "validation_1pct_genuine_fpr": low_fpr_results,
          "manipulation_type_rates": by_type, "paired_auc_differences": auc_differences,
          "coverage": coverage, "runtime": runtimes, "findings": findings}
for name, frame in tables.items():
    path = run_dir / f"{name}.csv"
    temporary = path.with_suffix(".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)
write_json(run_dir / "configuration.json", {**configuration, "execution_status": "complete_validation",
           "completed_at_utc": datetime.now(timezone.utc).isoformat(), "timings_seconds": timings})
print(f"Saved current-run validation results to {run_dir.relative_to(PROJECT_ROOT)}")
display(Markdown("**Conclusion:** candidate labels above concern these fixed-classifier validation experiments only. "
                 "Whether they help the full model remains untested. Freeze selected features, model settings and "
                 "thresholds before any final-test evaluation."))


,group,hypothesis,target,AUC,AUC_change_vs_control,CI_low,CI_high,recall_at_1pct_genuine_FPR,recall_change_vs_control,status
0,phrases,Caption details disagree with the image,image_fake,0.5873,0.0039,0.0025,0.0054,0.0209,-0.0001,candidate
1,phrases,Caption details disagree with the image,text_fake,0.8453,0.0193,0.0168,0.0220,0.4079,0.0552,candidate
2,phrases,Caption details disagree with the image,is_fake,0.6884,0.0083,0.0068,0.0098,0.1751,0.0209,candidate
3,names,Named people disagree with visible faces,image_fake,0.5924,0.0099,0.0070,0.0129,0.0216,-0.0012,candidate
4,names,Named people disagree with visible faces,text_fake,0.8315,0.0009,-0.0001,0.0018,0.3950,0.0051,inconclusive
5,names,Named people disagree with visible faces,is_fake,0.6890,0.0034,0.0020,0.0047,0.1731,0.0048,candidate
6,expression,Facial valence and caption sentiment interact,image_fake,0.6325,0.0015,0.0006,0.0024,0.0463,0.0024,candidate
7,expression,Facial valence and caption sentiment interact,text_fake,0.8321,0.0068,0.0053,0.0080,0.3287,-0.0104,candidate
8,expression,Facial valence and caption sentiment interact,is_fake,0.7078,0.0023,0.0013,0.0034,0.1456,-0.0044,candidate


,split,group,rows,features,available_fraction,finite
0,train_sample,phrases,49943,14,1.000000,True
1,train_sample,names,49943,7,0.797029,True
2,train_sample,expression,49943,15,1.000000,True
3,validation,phrases,22126,14,1.000000,True
4,validation,names,22126,7,0.785501,True
5,validation,expression,22126,15,1.000000,True
6,test,phrases,50705,14,1.000000,True
7,test,names,50705,7,0.816310,True
8,test,expression,50705,15,1.000000,True


,experiment,seconds
0,Original: global CLIP,2.33
1,Original: global + phrase control,2.27
2,Original: global + phrases,2.37
3,Expanded: baseline,2.27
4,Expanded: phrases,2.50
5,Expanded: names,2.55
6,Expanded: expression,2.43
7,Expanded: phrases + names,2.66
8,Expanded: phrases + expression,2.55
9,Expanded: names + expression,2.65


Saved current-run validation results to results/multimodal_features/1aeaeb7047243640


**Conclusion:** candidate labels above concern these fixed-classifier validation experiments only. Whether they help the full model remains untested. Freeze selected features, model settings and thresholds before any final-test evaluation.

# Summary

| Saved group | Relationship measured | Main limitation |
|---|---|---|
| `mm_phrase` | Caption phrases versus whole image | Parsing errors and invisible context |
| `mm_name_face` | Named people versus individual faces | CLIP name recognition is incomplete |
| `mm_expression` | CLIP facial-valence proxy versus caption sentiment | Emotional agreement is not required for genuine news |

All exports contain `row_id` plus finite feature columns, with availability indicators for missing evidence. Findings, controls and uncertainty are recorded above for each tested group; negative results are retained.

**Final-model candidates:** use the measured findings table to identify groups worth testing, checking low-FPR behaviour as well as AUC. `05` is unchanged. A future `05a` can compare its reference architecture with candidate features under identical training and evaluation settings. These results do not yet establish improvement over that model, and no final-test evaluation is performed here.
